# 05 · Feature Extraction — Colab (T4)

GPU execution path for notebook 05. Same logic, but on a Tesla T4 with the images staged to local disk first.

## In Drive before running
```
DFU_project/
  data/interim/folds_severity.csv
  data/interim/folds_infection.csv
  data/interim/labels_raw.csv
  images.zip          <- all referenced images, zipped
```

Run cells in order. Cell 3 (copy to local disk) is the one that matters for speed: reading images straight from Drive is 200-400ms each and wipes out the GPU advantage.

In [ ]:
# Cell 1 · GPU check
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True,text=True).stdout.strip() or 'no GPU')
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('cuda ready')

In [ ]:
# Cell 2 · mount Drive
from google.colab import drive; drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/DFU_project')   # edit to match
assert DRIVE.exists(), f'{DRIVE} not found'
print('found', DRIVE)

In [ ]:
# Cell 3 · stage images and CSVs on local SSD
# images.zip is built LOCALLY by make_images_zip.py, which includes only
# the images the fold CSVs reference, deduplicated, and keeps the two
# corpora in separate roboflow/ and dfuc/ subfolders so a filename used
# by both (both corpora share the same "<id>_jpg.rf.<hash>.jpg" naming
# convention) cannot overwrite the other.
import shutil, time, zipfile
LOCAL = Path('/content/work'); LOCAL.mkdir(exist_ok=True)
(LOCAL/'data/interim').mkdir(parents=True, exist_ok=True)
(LOCAL/'data/features').mkdir(parents=True, exist_ok=True)

for f in ['folds_severity.csv', 'folds_infection.csv', 'labels_raw.csv']:
    shutil.copy(DRIVE/'data/interim'/f, LOCAL/'data/interim'/f)

t0 = time.time()
shutil.copy(DRIVE/'images.zip', LOCAL/'images.zip')
with zipfile.ZipFile(LOCAL/'images.zip') as z:
    z.extractall(LOCAL/'images')

n_rf = sum(1 for _ in (LOCAL/'images/roboflow').glob('*.jpg'))
n_df = sum(1 for _ in (LOCAL/'images/dfuc').glob('*.jpg'))
print(f'staged in {time.time()-t0:.0f}s: {n_rf:,} roboflow, {n_df:,} dfuc')
assert n_rf > 0 and n_df > 0, (
    'expected roboflow/ and dfuc/ subfolders inside images.zip. '
    'Build it with make_images_zip.py, which namespaces both corpora.')

In [ ]:
# Cell 4 · rewrite CSV paths to local, by filename WITHIN each corpus
# Indexed separately per subfolder, not as one flat pool across both
# corpora. A flat index would let a DFUC filename silently shadow a
# Roboflow one that happens to match (both corpora share the same
# Roboflow-export naming convention).
import pandas as pd
from pathlib import Path

rf_index = {p.name: str(p) for p in (LOCAL/'images/roboflow').glob('*.jpg')}
df_index = {p.name: str(p) for p in (LOCAL/'images/dfuc').glob('*.jpg')}
print(f'indexed {len(rf_index):,} roboflow, {len(df_index):,} dfuc filenames')

# labels_raw.csv -> roboflow subfolder
lr = pd.read_csv(LOCAL/'data/interim/labels_raw.csv')
lr['path'] = lr['path'].map(lambda p: rf_index.get(Path(str(p)).name))
miss = int(lr.path.isna().sum())
print(f'labels_raw.csv: {miss} unmatched (roboflow)')
lr.to_csv(LOCAL/'data/interim/labels_raw.csv', index=False)

# folds_severity.csv -> roboflow subfolder, column "representative"
sv = pd.read_csv(LOCAL/'data/interim/folds_severity.csv')
sv['representative'] = sv['representative'].map(
    lambda p: rf_index.get(Path(str(p)).name))
miss = int(sv.representative.isna().sum())
print(f'folds_severity.csv: {miss} unmatched (roboflow)')
sv.to_csv(LOCAL/'data/interim/folds_severity.csv', index=False)

# folds_infection.csv -> dfuc subfolder, column "path"
inf = pd.read_csv(LOCAL/'data/interim/folds_infection.csv')
inf['path'] = inf['path'].map(lambda p: df_index.get(Path(str(p)).name))
miss = int(inf.path.isna().sum())
print(f'folds_infection.csv: {miss} unmatched (dfuc)')
assert miss == 0, (
    'unmatched DFUC images. Check images.zip was built from the same '
    'folds_infection.csv you are using here.')
inf.to_csv(LOCAL/'data/interim/folds_infection.csv', index=False)

In [ ]:
# Cell 5 · run the extraction logic from notebook 05
# The extraction code lives in ONE place: 05_feature_extraction.ipynb.
# Rather than copying it here (two copies drift apart), this cell converts
# that notebook to a script and runs it, so the local and Colab paths
# execute byte-identical logic.
#
# Requires 05_feature_extraction.ipynb uploaded to your Drive project folder.
import shutil, subprocess, sys, re
from pathlib import Path

%cd /content/work

src = DRIVE / '05_feature_extraction.ipynb'
assert src.exists(), (
    f'{src} not found. Upload 05_feature_extraction.ipynb to your Drive '
    'project folder next to the CSVs.')
shutil.copy(src, 'nb05.ipynb')

# convert to a script
# '--to python', not '--to script'. With '--to script' nbconvert infers
# the extension from notebook metadata and writes nb05.txt when the
# language spec is thin, which then fails to import. '--to python' always
# produces nb05.py.
subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'python',
                'nb05.ipynb', '--output', 'nb05'], check=True)

code = Path('nb05.py').read_text()

# Cell 1 of the local notebook re-checks inputs and calls raise SystemExit(1)
# if they are missing. Colab Cells 3 and 4 have already staged and verified
# those inputs, and SystemExit would kill this process, so drop that guard.
code = code.replace('raise SystemExit(1)', 'pass')

Path('nb05.py').write_text(code)
print('running extraction...\n')
subprocess.run([sys.executable, 'nb05.py'], check=True)

In [ ]:
# Cell 6 · save features back to Drive
import shutil
dst = DRIVE/'data/features'; dst.mkdir(parents=True, exist_ok=True)
for f in (LOCAL/'data/features').glob('*'):
    shutil.copy(f, dst/f.name)
print('features saved to', dst)
print('Colab wipes /content on disconnect — always run this before closing.')